# BioEng-TermBench: GPU benchmark (3D bioheat)

Runs the 3D version of the t04 bioheat solver on CPU (NumPy) and GPU (CuPy,
and a fused CUDA kernel), **checks correctness before timing**, and reports
throughput and speed-up.

Runtime: *Runtime > Change runtime type > T4 GPU* (free tier is enough).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 1. Get the code
Clone the repository (after it is pushed), or upload `bioeng-termbench.zip` with the file panel and unzip it.

In [ ]:
import os
if not os.path.exists("bioeng-termbench"):
    if os.path.exists("bioeng-termbench.zip"):
        !unzip -q bioeng-termbench.zip
    else:
        !git clone https://github.com/mbote-droid/bioeng-termbench.git
%cd bioeng-termbench
!pip install -q -e .
import cupy, numpy
print("cupy", cupy.__version__, "numpy", numpy.__version__)

## 2. Correctness gates
The GPU backends must reproduce NumPy on the same grid (same discretisation, so
float64 agrees to round-off), and the 3D solver must converge to the 1D
spherical reference at second order. These are the same checks as
`tests/unit/test_gpu_bioheat3d.py`, which also runs here with a GPU present.

In [ ]:
!python -m pytest -q tests/unit/test_gpu_bioheat3d.py

## 3. Benchmark
Throughput for grids from 65^3 to 257^3 nodes, float64 and float32, plus one
full treatment on the GPU. NumPy is timed up to 193^3 (larger takes minutes per
step set). Takes about 5-10 minutes on a T4.

In [ ]:
!python gpu/benchmark.py --sizes 64 128 192 256 --steps 40 --out gpu/results

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open("gpu/results/results.md").read()))
display(Image("gpu/results/speedup.png"))

## 4. Reading the results
* A 7-point stencil does about 15 floating-point operations per node and moves
  at least 2 doubles: it is **memory-bandwidth bound**. Expected speed-up is
  therefore near the ratio of memory bandwidths (T4 about 320 GB/s vs a few
  tens of GB/s for one CPU core), not the ratio of peak FLOPs.
* `cupy` (array expressions) launches one kernel per operation and writes
  every temporary to memory; `cupy-fused` does the whole Heun stage in one
  pass, so it moves far less data. The gap between them is the value of
  kernel fusion.
* The T4 runs float64 at 1/32 of its float32 rate. For the array-style
  `cupy` backend, which is bandwidth bound, float32 gains about 2x (half the
  bytes). The fused kernel is different: in float32 it reaches about 75 %
  of peak bandwidth, but in float64 its two exponentials and two divisions
  per node are limited by the slow double-precision units, so float32 is
  about 6-7x faster on a T4. Float32 is accurate enough here (temperatures
  to about 1e-4 K), but the correctness gates are run in float64.
* Small grids do not fill the GPU (launch overhead dominates); speed-up grows
  with grid size until memory bandwidth saturates.

## 5. Save the results
Download `gpu/results/` (results.json, results.md, speedup.png) and commit them
to the repository under `gpu/results/`, so the README can cite measured numbers.

In [ ]:
!cd gpu && zip -qr results.zip results && ls -la results.zip
from google.colab import files
files.download("gpu/results.zip")